<div style="padding: 20px; background: linear-gradient(90deg, #00b09b 0%, #96c93d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🛠️ Module 3.4: The Free RAG Stack (LangChain + Groq)</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Implementing zero-cost local embeddings with blazing-fast LLM inference.</p>
</div>

---

## 1. LangChain's Unified Interface

LangChain makes it incredibly easy to swap out embedding models. By using `HuggingFaceEmbeddings`, we get access to thousands of free local models without changing the rest of our RAG architecture.

> [!IMPORTANT]
> - `embed_documents(List[str])`: Use this when encoding your database/documents.
> - `embed_query(str)`: Use this when encoding the user's question. Some models expect queries to be prefixed with 'query:'.

In [4]:
# Load environment variables from .env file
from dotenv import load_dotenv
import os
load_dotenv()

# 1. Initialize the free local embedding model via LangChain
from langchain_huggingface import HuggingFaceEmbeddings
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("LangChain Local Embeddings Initialized.")

# 2. Demonstrate document vs query encoding
docs = [
    "The Eiffel Tower is in Paris, France.",
    "The Great Wall of China is visible from space.",
    "Rome is the capital city of Italy."
]

doc_vecs = embeddings.embed_documents(docs)
query_vec = embeddings.embed_query("Where is the Eiffel Tower located?")

print(f"Encoded {len(doc_vecs)} documents (Dimensions: {len(doc_vecs[0])})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LangChain Local Embeddings Initialized.
Encoded 3 documents (Dimensions: 384)


## 2. A Mini Semantic Search Engine

Before we introduce complex Vector Databases (like Chroma or FAISS), let's write a simple semantic search function using raw math to see how retrieval actually works.

In [5]:
import numpy as np

def retrieve_top_k(query: str, k=1):
    # 1. Embed the query
    q_emb = embeddings.embed_query(query)
    
    # 2. Calculate Cosine Similarity with all documents
    scores = []
    for doc, d_emb in zip(docs, doc_vecs):
        sim = np.dot(q_emb, d_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(d_emb))
        scores.append(sim)
        
    # 3. Get the index of the highest score
    best_idx = np.argmax(scores)
    return docs[best_idx], scores[best_idx]

user_query = "Which city is the capital of Italy?"
retrieved_context, score = retrieve_top_k(user_query)

print(f"Query: '{user_query}'")
print(f"Retrieved Document: '{retrieved_context}' (Similarity Score: {score:.4f})")

Query: 'Which city is the capital of Italy?'
Retrieved Document: 'Rome is the capital city of Italy.' (Similarity Score: 0.8605)


## 3. The RAG Pipeline (Connecting to Groq)

Now we have the **Retrieval** part. The final step is **Generation**. We will use `langchain-groq` which offers a generous free tier and the fastest inference speeds on the market for Llama 3 models.

> [!NOTE]
> You must have a free API key from [console.groq.com](https://console.groq.com) saved in your environment as `GROQ_API_KEY` to run the following cell.

In [6]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# Ensure you have the GROQ_API_KEY set. 
# If you don't have one, this cell will fail gracefully.
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    # Initialize Groq LLM
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # Create a simple prompt template
    prompt_template = PromptTemplate.from_template(
        """Answer the question using ONLY the provided context.
        
        Context: {context}
        Question: {question}
        
        Answer:"""
    )
    
    # Generate the final answer
    chain = prompt_template | llm
    
    response = chain.invoke({
        "context": retrieved_context,
        "question": user_query
    })
    
    print("\n--- Final RAG Answer ---")
    print(response.content)
else:
    print("\n[SKIPPED] GROQ_API_KEY not found in environment.")
    print("Set your key and re-run to see the final Generation step in action!")


--- Final RAG Answer ---
Rome is the capital city of Italy.
